# House Stage 1 regression: WLS, rolling 2018–2022

Weighted least squares against OLS on the classic formula, in the same expanding-window backtest as `01_regression_2018to2022_eval.ipynb`. Each window refits every model on its own training years and scores the next election:

| eval year | train years |
|---|---|
| 2018 | 2006–2016 |
| 2020 | 2006–2018 |
| 2022 | 2006–2020 |

`run_wls` (`experiments/house/run.py`) is `run_ols` with one change: the fit is `smf.wls(formula, data=train, weights=training_weights(train, ...)).fit()`. The weights come from the training rows alone, after exclusions, and are recomputed and normalized to mean 1 in every window, so nothing from the eval year enters a fit. Every model in a window uses the same training and evaluation rows, and the evaluation subsets (valid, `COMPETITIVE`) are the original notebook's. 2024 is never used (`USE_HOLDOUT = False`).

In [10]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))  # repository root, for experiments.house
import house.regression as hr  # hr.<Tab> lists everything; names are in src/house/regression/__init__.py
from experiments.house import evaluation, tracking
from experiments.house.run import run_ols, run_wls, training_weights

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
import warnings

with warnings.catch_warnings(action="ignore"):  # loading warns with the same summaries printed below
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
    targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)
print(hr.validate_schema(predictors).summary())  # .issues lists every affected key

In [12]:
FORMULA = "adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed"
ROLLING_WINDOWS = {  # eval year: its expanding training window
    2018: [2006, 2008, 2010, 2012, 2014, 2016],
    2020: [2006, 2008, 2010, 2012, 2014, 2016, 2018],
    2022: [2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020],
}
USE_HOLDOUT = False     # never True here: no window touches HOLDOUT_YEARS (2024)
REQUIRE_COMPLETE = []   # extra columns a row must have; every model below uses the same formula, so the same rows
# Evaluation only (training is unaffected): observed, non-imputed targets of these race types are scored ("valid")
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
# Competitive subsets of the valid targets: name -> (column, threshold); a row is in one when abs(column) <= threshold.
# column is a column name or arithmetic of them, from the targets or the predictors
EX_ANTE_THRESHOLD = 10   # points: any non-negative int or float, e.g. 5, 7, or 7.5
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    # expected margin under uniform swing: two-party lean (2 x cook_pvi) + eve-of-election two-party generic ballot
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),  # observed two-party margin
}
SUMMARY_SECTIONS = {"ALL": "valid", "EX_ANTE_COMPETITIVE": "competitive_ex_ante",
                    "EX_POST_COMPETITIVE": "competitive_ex_post"}
RANK_BY = "mae"     # variants ranked by mean out-of-sample MAE ...
TIEBREAK = "rmse"   # ... ties broken by RMSE

## Weighting

`training_weights` multiplies three factors for a training row from election year *t*. Each is on or off per variant:

- **Equal-year:** w = 1 / n_t, where n_t is the number of valid training rows in year *t*. Every training year then carries the same total weight, whatever its number of usable districts.
- **Recency:** w = 2^(−(T − t)/H), where T is the window's latest training year. A year H years older counts half. Measuring from the eval year instead only rescales the weights, which normalization removes.
- **Election type:** w = `MIDTERM_WEIGHT` in midterm years (2006, 2010, …) and 1.0 in presidential years.

The weights are then normalized to mean 1 within the window. Rescaling doesn't change WLS estimates. Every variant is compared with OLS (all weights 1), which is the base of every Δ below.

In [13]:
HALF_LIVES = [6, 8, 12]  # recency half-lives, in years
MIDTERM_WEIGHT = 1.25    # midterm elections, against 1.0 for presidential ones

WLS_WEIGHTINGS = {  # variant: training_weights settings
    "wls_equal_year": {"equal_year": True, "half_life": None, "midterm_weight": 1.0},
    **{f"wls_recency_h{h}": {"equal_year": False, "half_life": h, "midterm_weight": 1.0} for h in HALF_LIVES},
    **{f"wls_equal_year_recency_h{h}": {"equal_year": True, "half_life": h, "midterm_weight": 1.0}
       for h in HALF_LIVES},
    "wls_election_type": {"equal_year": False, "half_life": None, "midterm_weight": MIDTERM_WEIGHT},
    **{f"wls_all_h{h}": {"equal_year": True, "half_life": h, "midterm_weight": MIDTERM_WEIGHT} for h in HALF_LIVES},
}
list(WLS_WEIGHTINGS)

['wls_equal_year',
 'wls_recency_h6',
 'wls_recency_h8',
 'wls_recency_h12',
 'wls_equal_year_recency_h6',
 'wls_equal_year_recency_h8',
 'wls_equal_year_recency_h12',
 'wls_election_type',
 'wls_all_h6',
 'wls_all_h8',
 'wls_all_h12']

In [14]:
# Every window: OLS (the base), then each WLS variant, all on the same rows
rolling = {}
for eval_year, train_years in ROLLING_WINDOWS.items():
    ols = run_ols(
        formula=FORMULA,
        train_years=train_years,
        eval_years=[eval_year],
        predict_years=[],
        predictors=predictors,
        targets=targets,
        use_holdout=USE_HOLDOUT,
        require_complete=REQUIRE_COMPLETE,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name="ols",
        save=False,
    )
    wls = [
        run_wls(
            formula=FORMULA,
            train_years=train_years,
            eval_years=[eval_year],
            predict_years=[],
            equal_year=weights["equal_year"],
            half_life=weights["half_life"],
            midterm_weight=weights["midterm_weight"],
            predictors=predictors,
            targets=targets,
            use_holdout=USE_HOLDOUT,
            require_complete=REQUIRE_COMPLETE,
            race_types=EVAL_RACE_TYPES,
            competitive=COMPETITIVE,
            name=name,
            save=False,
        )
        for name, weights in WLS_WEIGHTINGS.items()
    ]
    rolling[eval_year] = {"ols_vs_wls": [ols, *wls]}  # one block, OLS first: the base of every delta

In [15]:
# Each training year's share of the total weight (%), in each window: what every variant actually does to the sample
shares = {}
for eval_year, blocks in rolling.items():
    train = blocks["ols_vs_wls"][0].predictions.query("role == 'train'")[["year"]]
    for name, weights in {"ols": {}, **WLS_WEIGHTINGS}.items():
        share = training_weights(train, **weights).groupby(train["year"]).sum()
        shares[(eval_year, name)] = 100 * share / share.sum()
pd.DataFrame(shares).rename_axis(columns=["eval_year", "variant"]).round(1)

eval_year  2018                                                               \
variant     ols wls_equal_year wls_recency_h6 wls_recency_h8 wls_recency_h12   
year                                                                           
2006       16.6           16.7            8.7           10.4            12.2   
2008       16.6           16.7           10.9           12.3            13.7   
2010       17.8           16.7           14.8           15.7            16.5   
2012       17.1           16.7           17.9           17.9            17.8   
2014       15.7           16.7           20.7           19.6            18.4   
2016       16.3           16.7           27.0           24.1            21.4   
2018        NaN            NaN            NaN            NaN             NaN   
2020        NaN            NaN            NaN            NaN             NaN   

eval_year                                                      \
variant   wls_equal_year_recency_h6 wls_equal_year_recency_h8   
year                                                            
2006                            8.7                      10.3   
2008                           10.9                      12.3   
2010                           13.8                      14.6   
2012                           17.3                      17.4   
2014                           21.8                      20.7   
2016                           27.5                      24.6   
2018                            NaN                       NaN   
2020                            NaN                       NaN   

eval_year                                                          ...  \
variant   wls_equal_year_recency_h12 wls_election_type wls_all_h6  ...   
year                                                               ...   
2006                            12.2              18.5        9.8  ...   
2008                            13.7              14.7        9.8  ...   
2010                            15.4              19.8       15.5  ...   
2012                            17.3              15.2       15.6  ...   
2014                            19.4              17.4       24.6  ...   
2016                            21.8              14.4       24.8  ...   
2018                             NaN               NaN        NaN  ...   
2020                             NaN               NaN        NaN  ...   

eval_year           2022                                 \
variant   wls_recency_h6 wls_recency_h8 wls_recency_h12   
year                                                      
2006                 4.7            6.2             7.9   
2008                 6.0            7.3             8.8   
2010                 8.1            9.3            10.7   
2012                 9.8           10.7            11.5   
2014                11.3           11.7            11.8   
2016                14.7           14.4            13.8   
2018                19.7           18.1            16.4   
2020                25.7           22.3            19.1   

eval_year                                                      \
variant   wls_equal_year_recency_h6 wls_equal_year_recency_h8   
year                                                            
2006                            4.9                       6.3   
2008                            6.1                       7.5   
2010                            7.7                       8.9   
2012                            9.7                      10.6   
2014                           12.2                      12.6   
2016                           15.4                      15.0   
2018                           19.4                      17.8   
2020                           24.5                      21.2   

eval_year                                                                     \
variant   wls_equal_year_recency_h12 wls_election_type wls_all_h6 wls_all_h8   
year                                                          

## Results

For each section (ALL, EX_ANTE and EX_POST on the original notebook's definitions), there are two tables. The first is **by eval year**, each window on its own. The second is the **mean over 2018, 2020 and 2022**, each window weighted equally. Variants are ranked by mean MAE, with ties broken by RMSE.

- **Metrics:** `n_train`, `n`, MAE, RMSE, bias (mean residual, actual − predicted: + means Democrats outperformed the forecast), accuracy, precision D/R and misses.
- **Mean table extras:** `|bias|` (the mean of each window's |bias|, so opposite misses don't cancel) and `mae_<year>` for each window.
- **Δ columns:** variant − OLS in the same window, on the same rows. ✓ (green) marks an improvement and ✗ (red) a regression: lower MAE, RMSE, |bias| and misses, higher accuracy and precision.

In [16]:
VARIANTS = len(WLS_WEIGHTINGS)  # show every variant, ranked

for section, subset in SUMMARY_SECTIONS.items():
    by_year = pd.concat(
        {year: tracking.best_of_blocks(blocks, subset=subset, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)[0]
         for year, blocks in rolling.items()}, names=["eval_year"]).droplevel("block")
    mean, _ = tracking.rolling_best_of(rolling, subset=subset, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)
    display(Markdown(f"### {section} (`{subset}`): by eval year"))
    display(tracking.style_best_of(by_year))
    display(Markdown(f"### {section} (`{subset}`): mean of {', '.join(map(str, ROLLING_WINDOWS))}, "
                     f"ranked by {RANK_BY} (ties: {TIEBREAK})"))
    display(tracking.style_best_of(mean.droplevel("block")))

### ALL (`valid`): by eval year

### ALL (`valid`): mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
ols,2680,400,7.31,8.91,-1.03,2.59,94.89,96.35,94.22,20,5.11,7.72,7.93,6.27,-,-,-,-,-,-,-,-
wls_all_h6,2680,400,6.70,8.25,-0.84,2.74,94.81,96.35,94.05,21,5.19,7.35,7.27,5.49,-0.60 ✓,-0.66 ✓,-0.08 ✗,-0.01 ✗,-0.16 ✗,+0 ✗,+0.08 ✗,+0.14 ✗
wls_recency_h6,2680,400,6.74,8.28,-0.96,2.69,94.81,96.35,94.05,21,5.19,7.38,7.39,5.45,-0.57 ✓,-0.63 ✓,-0.08 ✗,-0.01 ✗,-0.16 ✗,+0 ✗,+0.08 ✗,+0.10 ✗
wls_equal_year_recency_h6,2680,400,6.74,8.28,-0.94,2.70,94.81,96.35,94.05,21,5.19,7.36,7.39,5.49,-0.56 ✓,-0.62 ✓,-0.08 ✗,-0.01 ✗,-0.16 ✗,+0 ✗,+0.08 ✗,+0.11 ✗
wls_all_h8,2680,400,6.83,8.39,-0.86,2.71,94.81,96.35,94.05,21,5.19,7.43,7.40,5.65,-0.48 ✓,-0.52 ✓,-0.08 ✗,-0.01 ✗,-0.16 ✗,+0 ✗,+0.08 ✗,+0.11 ✗
wls_recency_h8,2680,400,6.86,8.41,-0.98,2.66,94.81,96.35,94.05,21,5.19,7.45,7.52,5.61,-0.44 ✓,-0.49 ✓,-0.08 ✗,-0.01 ✗,-0.16 ✗,+0 ✗,+0.08 ✗,+0.07 ✗
wls_equal_year_recency_h8,2680,400,6.87,8.42,-0.96,2.67,94.81,96.35,94.05,21,5.19,7.44,7.52,5.65,-0.44 ✓,-0.49 ✓,-0.08 ✗,-0.01 ✗,-0.16 ✗,+0 ✗,+0.08 ✗,+0.08 ✗
wls_all_h12,2680,400,6.97,8.54,-0.87,2.68,94.81,96.35,94.05,21,5.19,7.52,7.53,5.85,-0.34 ✓,-0.37 ✓,-0.08 ✗,-0.01 ✗,-0.16 ✗,+0 ✗,+0.08 ✗,+0.09 ✗
wls_recency_h12,2680,400,7.00,8.57,-1.00,2.64,94.81,96.35,94.05,21,5.19,7.54,7.65,5.81,-0.31 ✓,-0.34 ✓,-0.08 ✗,-0.01 ✗,-0.16 ✗,+0 ✗,+0.08 ✗,+0.05 ✗


### EX_ANTE_COMPETITIVE (`competitive_ex_ante`): by eval year

### EX_ANTE_COMPETITIVE (`competitive_ex_ante`): mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
ols,2680,91,7.93,9.24,+0.91,4.18,82.90,88.14,83.16,16,17.10,9.47,7.95,6.36,-,-,-,-,-,-,-,-
wls_recency_h6,2680,91,7.05,8.27,+1.05,3.65,82.90,88.14,83.16,16,17.10,8.70,6.98,5.46,-0.88 ✓,-0.97 ✓,+0.00,+0.00,+0.00,+0,+0.00,-0.53 ✓
wls_equal_year_recency_h6,2680,91,7.06,8.28,+1.06,3.64,82.90,88.14,83.16,16,17.10,8.70,6.99,5.48,-0.87 ✓,-0.96 ✓,+0.00,+0.00,+0.00,+0,+0.00,-0.54 ✓
wls_all_h6,2680,91,7.07,8.30,+1.14,3.68,82.90,88.14,83.16,16,17.10,8.82,6.90,5.49,-0.86 ✓,-0.95 ✓,+0.00,+0.00,+0.00,+0,+0.00,-0.50 ✓
wls_recency_h8,2680,91,7.23,8.47,+1.01,3.75,82.90,88.14,83.16,16,17.10,8.87,7.21,5.60,-0.70 ✓,-0.77 ✓,+0.00,+0.00,+0.00,+0,+0.00,-0.43 ✓
wls_equal_year_recency_h8,2680,91,7.24,8.49,+1.02,3.75,82.90,88.14,83.16,16,17.10,8.87,7.22,5.62,-0.69 ✓,-0.75 ✓,+0.00,+0.00,+0.00,+0,+0.00,-0.43 ✓
wls_all_h8,2680,91,7.25,8.51,+1.11,3.79,82.90,88.14,83.16,16,17.10,9.01,7.12,5.63,-0.67 ✓,-0.73 ✓,+0.00,+0.00,+0.00,+0,+0.00,-0.39 ✓
wls_recency_h12,2680,91,7.43,8.71,+0.97,3.89,82.90,88.14,83.16,16,17.10,9.06,7.45,5.79,-0.49 ✓,-0.53 ✓,+0.00,+0.00,+0.00,+0,+0.00,-0.29 ✓
wls_equal_year_recency_h12,2680,91,7.45,8.73,+0.98,3.89,82.90,88.14,83.16,16,17.10,9.07,7.46,5.82,-0.47 ✓,-0.52 ✓,+0.00,+0.00,+0.00,+0,+0.00,-0.29 ✓


### EX_POST_COMPETITIVE (`competitive_ex_post`): by eval year

### EX_POST_COMPETITIVE (`competitive_ex_post`): mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
ols,2680,60,10.56,12.20,+0.86,6.55,69.93,80.00,74.20,18,30.07,12.25,11.88,7.54,-,-,-,-,-,-,-,-
wls_recency_h6,2680,60,9.50,11.07,+1.21,6.03,69.29,79.69,73.21,19,30.71,11.58,10.38,6.52,-1.06 ✓,-1.13 ✓,-0.64 ✗,-0.31 ✗,-0.99 ✗,+0 ✗,+0.64 ✗,-0.52 ✓
wls_all_h6,2680,60,9.51,11.10,+1.32,6.05,69.29,79.69,73.21,19,30.71,11.72,10.25,6.56,-1.05 ✓,-1.10 ✓,-0.64 ✗,-0.31 ✗,-0.99 ✗,+0 ✗,+0.64 ✗,-0.50 ✓
wls_equal_year_recency_h6,2680,60,9.52,11.10,+1.21,6.03,69.29,79.69,73.21,19,30.71,11.59,10.40,6.56,-1.04 ✓,-1.10 ✓,-0.64 ✗,-0.31 ✗,-0.99 ✗,+0 ✗,+0.64 ✗,-0.52 ✓
wls_recency_h8,2680,60,9.74,11.33,+1.11,6.08,69.29,79.69,73.21,19,30.71,11.74,10.75,6.73,-0.82 ✓,-0.87 ✓,-0.64 ✗,-0.31 ✗,-0.99 ✗,+0 ✗,+0.64 ✗,-0.47 ✓
wls_equal_year_recency_h8,2680,60,9.76,11.35,+1.12,6.08,69.29,79.69,73.21,19,30.71,11.74,10.76,6.76,-0.80 ✓,-0.85 ✓,-0.64 ✗,-0.31 ✗,-0.99 ✗,+0 ✗,+0.64 ✗,-0.47 ✓
wls_all_h8,2680,60,9.76,11.36,+1.23,6.10,69.29,79.69,73.21,19,30.71,11.88,10.62,6.77,-0.80 ✓,-0.84 ✓,-0.64 ✗,-0.31 ✗,-0.99 ✗,+0 ✗,+0.64 ✗,-0.45 ✓
wls_recency_h12,2680,60,9.99,11.60,+1.03,6.13,69.29,79.69,73.21,19,30.71,11.90,11.12,6.96,-0.56 ✓,-0.60 ✓,-0.64 ✗,-0.31 ✗,-0.99 ✗,+0 ✗,+0.64 ✗,-0.42 ✓
wls_equal_year_recency_h12,2680,60,10.01,11.62,+1.03,6.13,69.29,79.69,73.21,19,30.71,11.91,11.14,7.00,-0.54 ✓,-0.58 ✓,-0.64 ✗,-0.31 ✗,-0.99 ✗,+0 ✗,+0.64 ✗,-0.42 ✓


## Coefficients

The fitted coefficients of every model, by window and averaged over the three. Weighting moves the estimates through how much each training year counts. Incumbency (`inc_signed`) is shown on its own, with its change from OLS.

In [17]:
# each model's coefficients averaged over the windows (rows: models, columns: terms)
_, mean_coefficients = tracking.rolling_best_of(rolling, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)
mean_coefficients.droplevel("block").round(3)

,Intercept,gen_margin_raw,partisan_lean,inc_signed
run,,,,
ols,-0.228,0.865,0.811,12.776
wls_all_h6,-0.570,0.892,0.841,10.858
wls_recency_h6,-0.497,0.889,0.844,10.831
wls_equal_year_recency_h6,-0.529,0.893,0.843,10.877
wls_all_h8,-0.499,0.884,0.833,11.327
wls_recency_h8,-0.417,0.882,0.837,11.291
wls_equal_year_recency_h8,-0.446,0.886,0.836,11.337
wls_all_h12,-0.436,0.877,0.825,11.818
wls_recency_h12,-0.344,0.876,0.829,11.773


In [18]:
# the incumbency coefficient (points of margin per unit of inc_signed: +1 D incumbent, -1 R, 0 open), by window
incumbency = pd.DataFrame({
    eval_year: {run.config["name"]: run.coefficients.loc["inc_signed", "coef"] for run in blocks["ols_vs_wls"]}
    for eval_year, blocks in rolling.items()
}).rename_axis(index="model", columns="eval_year")
incumbency["mean"] = incumbency.mean(axis=1)
incumbency["Δmean vs ols"] = incumbency["mean"] - incumbency.loc["ols", "mean"]
incumbency.round(3)

eval_year,2018,2020,2022,mean,Δmean vs ols
model,,,,,
ols,13.811,12.798,11.718,12.776,0.000
wls_equal_year,13.825,12.837,11.810,12.824,0.048
wls_recency_h6,12.527,10.857,9.109,10.831,-1.945
wls_recency_h8,12.825,11.325,9.723,11.291,-1.485
wls_recency_h12,13.141,11.808,10.371,11.773,-1.002
wls_equal_year_recency_h6,12.515,10.896,9.219,10.877,-1.899
wls_equal_year_recency_h8,12.819,11.362,9.829,11.337,-1.439
wls_equal_year_recency_h12,13.140,11.845,10.474,11.819,-0.956
wls_election_type,13.854,12.737,11.782,12.791,0.015
